# A 15-minute free-response drill bot in ANSI C

`dgs/c/frqbot/frqbot.c` is one C89 file: `gcc -std=c89 -pedantic -Wall -Wextra -Werror`. It
asks original practice questions in **ANSI C, laser safety, AP Biology, AP Chemistry,
AP Psychology, AP Economics (supply and demand), and CompTIA**, and gives each question a
15-minute limit. A late answer scores 0, as on the exam.

**Use it in a terminal** (that's where the timer is real):
```
gcc -std=c89 -pedantic -Wall -Wextra -Werror dgs/c/frqbot/frqbot.c -o frqbot -lm
./frqbot --subject econ          # or c, laser, bio, chem, psych, comptia
./frqbot --n 4 --seed 7          # 4 random questions = one hour of practice
```
The notebook below feeds the bot answers automatically, to show how its grading works.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
import pandas as pd
from IPython.display import display
from dgs import frqbot as B
from dgs.eventsim import ToolUnavailable
pd.set_option("display.max_colwidth", 95)
try:
    display(pd.DataFrame(B.bank(), columns=["subject", "question"])); HAVE = True
except ToolUnavailable as err:
    HAVE = False; print("skipped:", err)

,subject,question
0,c,"int a = 7, b = 2; what does printf(""%d"", a / b * b + a % b) print?"
1,c,Why does the body of if (x = 0) { ... } never run?
2,laser,Eyewear is rated OD 4 at the laser's wavelength. What fraction of the light gets through?
3,laser,The exposure is 50 W/cm^2 and the MPE is 2.5e-3 W/cm^2. What minimum OD is needed?
4,bio,16% of a population shows a recessive trait (Hardy-Weinberg). What fraction are carriers (2...
5,bio,Explain why an enzyme stops working at very high temperature.
6,chem,What is the pH of 0.0010 M HCl (a strong acid)?
7,chem,How many moles are in 36.0 g of water (molar mass 18.02 g/mol)?
8,psych,"After Pavlov's training, the bell alone makes the dog salivate. Name the role the bell plays."
9,psych,Define negative reinforcement and say what it does to a behavior.


## How it grades

- **Numbers:** a relative tolerance (1% for most). `2.00` and `2` both count; `25` for 20 doesn't.
- **Written answers:** the key ideas must appear, and `a|b` means either word counts. The bot
  tells you which idea was **missing**. Keyword grading is crude. A real reader also checks that
  your reasoning connects the ideas, so write full sentences.

In [2]:
if HAVE:
    def shaky(prompt):                      # some right, some half-right, some wrong
        if "Demand increases" in prompt: return "the price goes up"
        if "equilibrium price." in prompt: return "20"
        if "price ceiling" in prompt: return "25"
        return "not sure"
    res, (score, asked) = B.drill(shaky, subject="econ", seed=3)
    display(pd.DataFrame(res)[["prompt", "verdict", "detail", "why"]]); print(f"score {score}/{asked}")

,prompt,verdict,detail,why
0,"Demand Qd = 100 - 2P, supply Qs = 20 + 2P. Find the equilibrium price.",CORRECT,"CORRECT (you: 20, answer: 20, 0 s)","Set Qd = Qs: 100 - 2P = 20 + 2P -> 4P = 80 -> P = 20, Q = 60."
1,Demand increases while supply stays the same. What happens to the equilibrium price and qua...,NOT YET,"NOT YET (2 of 3 ideas (need 3); missing: [quantity], 0 s)",Demand shifts right along a fixed supply curve: equilibrium price and quantity both rise.
2,"Qd = 100 - 2P, Qs = 20 + 2P, and a price ceiling is set at P = 15. How large is the shortage?",NOT YET,"NOT YET (you: 25, answer: 20, 0 s)","Qd = 100 - 30 = 70, Qs = 20 + 30 = 50; shortage = 70 - 50 = 20 units."


score 1/3


## The timer (here with a 1-second limit and a 2-second answer)

In [3]:
if HAVE:
    res, (score, asked) = B.drill(B.model_answer, subject="comptia", n=1, limit=1, seed=1, delay_s=2.2)
    print(res[0]["detail"]); print("score", score, "/", asked)

TIME: 2 s > 1 s limit, scored 0 (3 of 3 ideas (need 3); missing: none)
score 0 / 1


## Full run with model answers: every subject

In [4]:
if HAVE:
    res, (score, asked) = B.drill(B.model_answer, seed=11)
    df = pd.DataFrame(res)
    display(df.groupby("subject")["verdict"].apply(lambda v: f"{(v == 'CORRECT').sum()}/{len(v)}").rename("correct"))
    print(f"total {score}/{asked}")

subject
bio        2/2
c          2/2
chem       2/2
comptia    3/3
econ       3/3
laser      2/2
psych      2/2
Name: correct, dtype: object

total 16/16
